In [1]:
# Se importan las librerias
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from datetime import datetime
# Obtener la fecha y hora actual
ahora = datetime.now()

# Extraer el año
año = ahora.year


In [2]:
# Se lee el archivo de novedades
df_excel=pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\TELECAMPUS_RE\NOVEDADES_JULIO_2026.xlsx', sheet_name='Hoja1')
df_excel = df_excel.drop_duplicates()
df_excel['IDENTIFICACION']=df_excel['IDENTIFICACION'].astype(str)
df_excel.shape

(2778, 10)

In [3]:
df_excel

,MES,TELECAMPUS,IDENTIFICACION,NOMBRE ESTUDIANTE,PERIODO,TIPO ESTUDIANTE,TIPO,FECHA DE PAGO,VALOR DE RECAUDO,OBSERVACION TELECAMPUS
0,Julio,telecampus bosa,1056770119,aaa,2022B,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion
1,Julio,telecampus bosa,1002249342,aaa,22V04,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion
2,Julio,telecampus bosa,79828435,aaa,22V03,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion
3,Julio,telecampus bosa,1000380934,aaa,2022B,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion
4,Julio,telecampus bosa,1012327564,aaa,2022B,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion
...,...,...,...,...,...,...,...,...,...,...
2775,Julio,telecampus bosa,1125473998,aaa,25V06,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion
2776,Julio,telecampus bosa,1012406529,aaa,25V06,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion
2777,Julio,telecampus bosa,1000160212,aaa,25V06,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion
2778,Julio,telecampus bosa,1031164972,aaa,25V06,Nuevo,Pregrado,2026-07-15,123456,no esta en la liquidacion


In [4]:
# Se crea otro dataframe de las columnas a utilizar
df_data = df_excel[['IDENTIFICACION','PERIODO','TELECAMPUS']]
df_data = df_data.rename(columns={'IDENTIFICACION':'DOC_ALUM','TELECAMPUS':'TELECAMPUS DE NOVEDAD'})
df_data.shape

(2778, 3)

In [5]:
# Se saca la lista de documentos y periodos
para_orden = df_data['DOC_ALUM'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
para_orden_per = df_data['PERIODO'].unique()
valores_unicos_df1_orden_per = "', '".join(map(str, para_orden_per))
valores_unicos_df1_orden_per = f"'{valores_unicos_df1_orden_per}'"

In [42]:
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    recibos_cod_uni= pd.read_sql(f'''
                             
        SELECT DISTINCT DOC_ALUM,PERIODO, NUEVO, CLASE_ACTUAL, FUERZAZOHO, ESTADO_PAGO, EST_MATRICULADO, TIPO, NOM_PROGRAMA,PROPIETARIO_ZOHO
                    FROM CUN_REPOSITORIO.zoho.base_personas
            WHERE doc_alum IN ({valores_unicos_df1_orden})
            AND DOCUMENTO IN ('FAMA','FECO')
            and PERIODO IN ({valores_unicos_df1_orden_per})
            --AND ANIO = ({año});

                               ''', engine2_1)
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")



Conexion a la base de datos realizada
Conexión cerrada


In [43]:
recibos_cod_uni2 = recibos_cod_uni.copy()
recibos_cod_uni2 = recibos_cod_uni2.map(lambda x: x.upper() if isinstance(x, str) else x)
recibos_cod_uni2['LLAVE'] = recibos_cod_uni2['DOC_ALUM'].astype(str) + recibos_cod_uni2['PERIODO']
recibos_cod_uni2.shape

(8002, 11)

In [44]:
df_sinpropietario = recibos_cod_uni2[recibos_cod_uni2['PROPIETARIO_ZOHO'].isna()].copy()
df_sinpropietario = df_sinpropietario[
    (df_sinpropietario['TIPO'] == 'POSTGRADO') & 
    (df_sinpropietario['NUEVO'] == 'ANTIGUO') 
]

df_sinpropietario.shape

(331, 11)

In [45]:
df_conpropietario = recibos_cod_uni2[recibos_cod_uni2['PROPIETARIO_ZOHO'].notna()].copy()
df_conpropietario = df_conpropietario[
    (df_conpropietario['TIPO'] == 'POSTGRADO') & 
    (df_conpropietario['NUEVO'] == 'NUEVO') 
]
df_conpropietario.shape

(232, 11)

In [46]:
df_con_sin_propietario = df_sinpropietario.merge(df_conpropietario[['DOC_ALUM','NOM_PROGRAMA','FUERZAZOHO','PROPIETARIO_ZOHO']],on=['DOC_ALUM','NOM_PROGRAMA'],how='left')
df_con_sin_propietario = df_con_sin_propietario.rename(columns={'PROPIETARIO_ZOHO_y':'PROPIETARIO_ZOHO_FINAL','FUERZAZOHO_y':'FUERZAZOHO_FINAL'})
df_con_sin_propietario = df_con_sin_propietario[['DOC_ALUM','PERIODO','PROPIETARIO_ZOHO_FINAL', 'FUERZAZOHO_FINAL']]
df_con_sin_propietario = df_con_sin_propietario[df_con_sin_propietario['PROPIETARIO_ZOHO_FINAL'].notna()]
df_con_sin_propietario

,DOC_ALUM,PERIODO,PROPIETARIO_ZOHO_FINAL,FUERZAZOHO_FINAL
0,1000145039,25ET1,BOGOTA- D.C.,TELECAMPUS
1,1000145039,25ET2,BOGOTA- D.C.,TELECAMPUS
2,1000268822,25ET4,TELECAMPUS BOSA,TELECAMPUS
3,1002686748,25ET1,BOGOTA- D.C.,TELECAMPUS
4,1003251304,25ET1,,TELECAMPUS
...,...,...,...,...
326,91527169,25ET2,TELECAMPUS BOSA,TELECAMPUS
327,91527169,25ET3,TELECAMPUS BOSA,TELECAMPUS
328,91527169,25ET4,TELECAMPUS BOSA,TELECAMPUS
329,94387232,25ET1,BOGOTA- D.C.,TELECAMPUS


In [47]:
df_final = recibos_cod_uni2.merge(df_con_sin_propietario,on=['DOC_ALUM','PERIODO'], how='left')
df_final['PROPIETARIO_ZOHO'] = df_final['PROPIETARIO_ZOHO'].fillna(df_final['PROPIETARIO_ZOHO_FINAL'])
df_final = df_final.drop(columns=['FUERZAZOHO_FINAL','PROPIETARIO_ZOHO_FINAL'])
df_final

,DOC_ALUM,PERIODO,NUEVO,CLASE_ACTUAL,FUERZAZOHO,ESTADO_PAGO,EST_MATRICULADO,TIPO,NOM_PROGRAMA,PROPIETARIO_ZOHO,LLAVE
0,1000001162,2026A,NUEVO,NUEVO,TELECAMPUS,PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,TELECAMPUS BOSA,10000011622026A
1,1000004469,24V02,NUEVO,NUEVO,TELECAMPUS,PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,BOGOTA- D.C.,100000446924V02
2,1000004727,25V05,NUEVO,NUEVO,TELECAMPUS,PAGO,1-ACTIVO,PREGRADO,PUBLICIDAD Y MERCADEO,TELECAMPUS BOSA,100000472725V05
3,1000007269,25V05,NUEVO,NUEVO,TELECAMPUS,PAGO,1-ACTIVO,PREGRADO,PUBLICIDAD Y MERCADEO,TELECAMPUS BOSA,100000726925V05
4,1000020199,24V06,NUEVO,NUEVO,TELECAMPUS,PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,,100002019924V06
...,...,...,...,...,...,...,...,...,...,...,...
7997,9910261,25V06,ANTIGUO,CONTINUO,PERMANENCIA,NO PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,NaN,991026125V06
7998,9910410,23V04,NUEVO,NUEVO,TELECAMPUS,PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,BOGOTA- D.C.,991041023V04
7999,9910410,23V05,ANTIGUO,NUEVO,PERMANENCIA,NO PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,,991041023V05
8000,9910410,24V01,ANTIGUO,CONTINUO,PERMANENCIA,NO PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,,991041024V01


In [48]:
df_pre_validos = df_final[
    (
        (df_final['FUERZAZOHO'].isin(['TELECAMPUS', 'PERMANENCIA']))
    ) &
    (
        (df_final['NUEVO'] == 'NUEVO') |
        (
            (df_final['NUEVO'] == 'ANTIGUO') &
            (df_final['TIPO'] == 'POSTGRADO')
        )
    ) &
    (df_final['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_final['ESTADO_PAGO'] == 'PAGO')
].copy()

df_pre_validos.shape

(2744, 11)

In [49]:
df_pre_novalidos = df_final[
    (
        (~df_final['FUERZAZOHO'].isin(['TELECAMPUS', 'PERMANENCIA']))
    ) |
    (
        (
            (df_final['NUEVO'] == 'ANTIGUO') &
            (df_final['TIPO'] == 'PREGRADO')
        )
    ) |
    (df_final['EST_MATRICULADO'] != '1-ACTIVO') |
    (df_final['ESTADO_PAGO'] != 'PAGO')
].copy()

df_pre_novalidos.shape

(5258, 11)

In [50]:

# Se asigna el motivo por el cual no pasa
df_pre_validos['ESTADO_FINAL'] = 'VALIDO'
df_pre_novalidos['ESTADO_FINAL'] = ''

# Condición 1
cond1 = ~df_pre_novalidos['FUERZAZOHO'].isin(['TELECAMPUS', 'PERMANENCIAS'])
df_pre_novalidos.loc[cond1, 'ESTADO_FINAL'] += 'Pertenece a otra fuerza, '

# Condición 2
cond2 = (
    (df_pre_novalidos['NUEVO'] == 'ANTIGUO') &
    (df_pre_novalidos['TIPO'] == 'PREGRADO')
)
df_pre_novalidos.loc[cond2, 'ESTADO_FINAL'] += 'Estudiante antiguo, '

# Condición 3
cond3 = df_pre_novalidos['EST_MATRICULADO'] != '1-ACTIVO'
df_pre_novalidos.loc[cond3, 'ESTADO_FINAL'] += 'Estado de matrícula diferente a 1-ACTIVO, '

# Condición 4
cond4 = df_pre_novalidos['ESTADO_PAGO'] != 'PAGO'
df_pre_novalidos.loc[cond4, 'ESTADO_FINAL'] += 'Estado no pago, '

# Quitar última coma
df_pre_novalidos['ESTADO_FINAL'] = (
    df_pre_novalidos['ESTADO_FINAL']
    .str.rstrip(', ')
)
df_pre_novalidos

,DOC_ALUM,PERIODO,NUEVO,CLASE_ACTUAL,FUERZAZOHO,ESTADO_PAGO,EST_MATRICULADO,TIPO,NOM_PROGRAMA,PROPIETARIO_ZOHO,LLAVE,ESTADO_FINAL
5,1000020199,25V03,ANTIGUO,CONTINUO,PERMANENCIA,PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,,100002019925V03,"Pertenece a otra fuerza, Estudiante antiguo"
6,1000020199,25V06,ANTIGUO,CONTINUO,PERMANENCIA,PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,NaN,100002019925V06,"Pertenece a otra fuerza, Estudiante antiguo"
8,1000020283,25V06,ANTIGUO,CONTINUO,PERMANENCIA,PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,NaN,100002028325V06,"Pertenece a otra fuerza, Estudiante antiguo"
10,1000035846,25V06,ANTIGUO,CONTINUO,PERMANENCIA,PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,NaN,100003584625V06,"Pertenece a otra fuerza, Estudiante antiguo"
13,1000048478,25V03,ANTIGUO,NUEVO,PERMANENCIA,NO PAGO,1-ACTIVO,PREGRADO,PUBLICIDAD Y MERCADEO,,100004847825V03,"Pertenece a otra fuerza, Estudiante antiguo, E..."
...,...,...,...,...,...,...,...,...,...,...,...,...
7996,9910261,25V04,ANTIGUO,NUEVO,PERMANENCIA,NO PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,,991026125V04,"Pertenece a otra fuerza, Estudiante antiguo, E..."
7997,9910261,25V06,ANTIGUO,CONTINUO,PERMANENCIA,NO PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,NaN,991026125V06,"Pertenece a otra fuerza, Estudiante antiguo, E..."
7999,9910410,23V05,ANTIGUO,NUEVO,PERMANENCIA,NO PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,,991041023V05,"Pertenece a otra fuerza, Estudiante antiguo, E..."
8000,9910410,24V01,ANTIGUO,CONTINUO,PERMANENCIA,NO PAGO,1-ACTIVO,PREGRADO,INGENIERIA DE SISTEMAS,,991041024V01,"Pertenece a otra fuerza, Estudiante antiguo, E..."


In [51]:
df_pre_novalidos_2 = df_data[~df_data['DOC_ALUM'].isin(df_pre_validos['DOC_ALUM'])].copy()
df_pre_novalidos_2 = df_pre_novalidos_2[~df_pre_novalidos_2['DOC_ALUM'].isin(df_pre_novalidos['DOC_ALUM'])].copy()
df_pre_novalidos_2['ESTADO_FINAL'] = 'NO CRUZA ICERBEG/SINU'
df_pre_novalidos_2

,DOC_ALUM,PERIODO,TELECAMPUS DE NOVEDAD,ESTADO_FINAL
94,72208476,22V06,telecampus bosa,NO CRUZA ICERBEG/SINU
95,1013104158,23T01,telecampus bosa,NO CRUZA ICERBEG/SINU
116,93138018,23V01,telecampus bosa,NO CRUZA ICERBEG/SINU
143,1031170574,23V01,telecampus bosa,NO CRUZA ICERBEG/SINU
148,1029141315,2023A,telecampus bosa,NO CRUZA ICERBEG/SINU
170,1013102366,23T01,telecampus bosa,NO CRUZA ICERBEG/SINU
174,1000588554,2023A,telecampus bosa,NO CRUZA ICERBEG/SINU
227,1026288737,23V02,telecampus bosa,NO CRUZA ICERBEG/SINU
238,1000124439,23V02,telecampus bosa,NO CRUZA ICERBEG/SINU
459,1019027395,23ET3,telecampus bosa,NO CRUZA ICERBEG/SINU


# SE VALIDA CON VINCULACIONES

In [52]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    df_vinculaciones = pd.read_sql("select * from CUN_REPOSITORIO.comisiones.COMISIONES_VINCULACIONES_ACTUALES", engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [53]:
df_vinculaciones['LLAVE_VINCULACIONES']= df_vinculaciones['Documento_Estudiante'].astype(str)+df_vinculaciones['PERIODO']
df_pre_validos['VINCULACIONES'] = df_pre_validos['LLAVE'].isin(df_vinculaciones['LLAVE_VINCULACIONES']).astype(int)
df_pre_validos.loc[df_pre_validos['VINCULACIONES'] == 1] 
df_pre_validos.shape

(2744, 13)

# SE VALIDA CON CONTACT

In [54]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    df_contact = pd.read_sql("SELECT * FROM CUN_REPOSITORIO.comisiones_contact.etiquetas WHERE EstadoFinal='OK'", engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")


Conexion a la base de datos realizada
Conexión cerrada


In [55]:
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_contact= pd.read_sql(f'''
                             
        SELECT * FROM CUN_REPOSITORIO.comisiones_contact.etiquetas WHERE EstadoFinal='OK'

                               ''', engine2_1)
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")


Conexion a la base de datos realizada
Conexión cerrada


In [56]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    df_contact2 = pd.read_sql("select * from CUN_REPOSITORIO.comisiones_contact.EtiquetasReliquidacion where EstadoFinal='OK'", engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [57]:
df_contact_total = pd.concat([df_contact, df_contact2], ignore_index=True)


In [58]:
df_contact_total['LLAVE_CONTACT']= df_contact_total['Doc_Estudiante'].astype(str)+df_contact_total['PERIODO']
df_pre_validos['CONTACT'] = df_pre_validos['LLAVE'].isin(df_contact_total['LLAVE_CONTACT']).astype(int)
df_pre_validos.loc[df_pre_validos['CONTACT'] == 1] 
df_pre_validos.shape

(2744, 14)

# SE VALIDA QUE LA OPORTUNIDAD CREADA EN ZOHO

In [59]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    df_oportunidades = pd.read_sql(f'''Select distinct Número_de_Documento,Periodo from zoho.CRM.Oportunidades where Fase ='Matriculado' 
                                    and Número_de_Documento in ({valores_unicos_df1_orden})''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [60]:
df_oportunidades['LLAVE_OPORTUNIDAD']= df_oportunidades['Número_de_Documento'].astype(str)+df_oportunidades['Periodo']
df_pre_validos['OPORTUNIDADES'] = df_pre_validos['LLAVE'].isin(df_oportunidades['LLAVE_OPORTUNIDAD']).astype(int)
df_pre_validos.loc[df_pre_validos['OPORTUNIDADES'] == 1] 
df_pre_validos.shape

(2744, 15)

# SE TRAEN LOS PERIODOS ACTIVOS

In [61]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    df_periodos = pd.read_sql(f'''SELECT PERIODO, TRY_CONVERT(date, FF_CLASE, 103) as fecha FROM CUN.MATRIZ_CALENDARIO
                                        WHERE AÑO = ({año}) ''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [62]:
# Obtener la fecha actual
import datetime
hoy = datetime.date.today()

# Calcular el mes anterior
if hoy.month == 1:
    mes_anterior = 12 
    anio_anterior = hoy.year - 1
else:
    mes_anterior = hoy.month -1
    anio_anterior = hoy.year

# Asignar variables con el mes anterior
MES_A_LIQUIDAR = f"{anio_anterior}/{mes_anterior}"
print("MES_A_LIQUIDAR =", MES_A_LIQUIDAR)

MES_A_LIQUIDAR = 2026/9


In [63]:
df_periodos['FECHA_FILTRO'] = pd.to_datetime(
    df_periodos['fecha'],
    errors='coerce'
).dt.date
fecha_max = pd.to_datetime(
    MES_A_LIQUIDAR + '/01',
    format='%Y/%m/%d'
).date()

df_periodos_activos = df_periodos[
    df_periodos['FECHA_FILTRO'] >= fecha_max
 
]
df_periodos_activos

,PERIODO,fecha,FECHA_FILTRO
2,2026C,2026-11-22,2026-11-22
3,2026D,2027-01-17,2027-01-17
6,26V03,2026-09-27,2026-09-27
7,26V04,2026-11-22,2026-11-22
8,26V05,2027-01-17,2027-01-17
9,26V06,2027-03-28,2027-03-28
12,26P03,2026-09-27,2026-09-27
13,26P04,2026-11-22,2026-11-22
14,26P05,2027-01-17,2027-01-17
15,26P06,2027-03-28,2027-03-28


In [64]:
df_pre_validos['PERIODO_ACTIVO'] = df_pre_validos['PERIODO'].isin(df_periodos_activos['PERIODO']).astype(int)
df_pre_validos.loc[df_pre_validos['PERIODO_ACTIVO'] == 1] 
df_pre_validos.shape

(2744, 16)

In [65]:
df_pre_validos['CONTACT'] = '0'

In [66]:
# Condición 1
cond1_1 = (
    (df_pre_validos['VINCULACIONES'] == '1') |
    (df_pre_validos['CONTACT'] == '1')
)

df_pre_validos.loc[cond1_1, 'ESTADO_FINAL'] += 'Ya se pago a otra fuerza, '


# Condición 2
# OPORTUNIDADES = 0
# PERO excluyendo POSTGRADO + ANTIGUO

cond2_2 = (
    (df_pre_validos['OPORTUNIDADES'] == '0') &
    ~(
        (df_pre_validos['NUEVO'] == 'ANTIGUO') &
        (df_pre_validos['TIPO'] == 'POSTGRADO')
    )
)

df_pre_validos.loc[cond2_2, 'ESTADO_FINAL'] += 'No esta ok en zoho, '


# Condición 3
cond3_3 = (
    (df_pre_validos['OPORTUNIDADES'] == '0') &
    (df_pre_validos['NUEVO'] == 'ANTIGUO') &
    (df_pre_validos['TIPO'] == 'POSTGRADO')
)

df_pre_validos.loc[cond3_3, 'ESTADO_FINAL'] += 'VALIDO, '


# Condición 4
cond4_4 = df_pre_validos['PERIODO_ACTIVO'] == '0'

df_pre_validos.loc[cond4_4, 'ESTADO_FINAL'] += 'Periodo Inactivo, '


# Limpiar coma final
df_pre_validos['ESTADO_FINAL'] = (
    df_pre_validos['ESTADO_FINAL']
    .str.rstrip(', ')
)

In [67]:
df_validos = df_pre_validos[df_pre_validos['ESTADO_FINAL'] == 'VALIDO']
df_validos = df_validos.drop_duplicates()
df_validos.shape

(2744, 16)

In [68]:
df_validos.columns

Index(['DOC_ALUM', 'PERIODO', 'NUEVO', 'CLASE_ACTUAL', 'FUERZAZOHO',
       'ESTADO_PAGO', 'EST_MATRICULADO', 'TIPO', 'NOM_PROGRAMA',
       'PROPIETARIO_ZOHO', 'LLAVE', 'ESTADO_FINAL', 'VINCULACIONES', 'CONTACT',
       'OPORTUNIDADES', 'PERIODO_ACTIVO'],
      dtype='object')

# SE UNIFICAN LAS VENTAS NO VALIDAS

In [69]:
df_novalidos = df_pre_validos[df_pre_validos['ESTADO_FINAL'] != 'VALIDO']
df_novalidos = df_novalidos.drop(columns=['VINCULACIONES', 'CONTACT','OPORTUNIDADES', 'PERIODO_ACTIVO'])
df_novalidos.shape

(0, 12)

In [70]:
df_novalidos_total = pd.concat([df_novalidos,df_pre_novalidos])
df_novalidos_total = df_novalidos_total.drop(columns='LLAVE')
df_novalidos_total.shape

(5258, 11)

In [71]:
df_novalidos_total = df_novalidos_total.merge(df_data[['DOC_ALUM','TELECAMPUS DE NOVEDAD']], how='inner',on='DOC_ALUM')
df_novalidos_total = df_novalidos_total.drop_duplicates()
df_novalidos_total.shape

(5258, 12)

In [72]:
df_pre_novalidos_2['NUEVO'] = ' '
df_pre_novalidos_2['CLASE_ACTUAL'] = ' '
df_pre_novalidos_2['FUERZAZOHO'] = ' '
df_pre_novalidos_2['ESTADO_PAGO'] = ' '
df_pre_novalidos_2['EST_MATRICULADO'] = ' '
df_pre_novalidos_2['TIPO'] = ' '
df_pre_novalidos_2['NOM_PROGRAMA'] = ' '
df_pre_novalidos_2['PROPIETARIO_ZOHO'] = ' '

In [73]:
df_novalidos_total_total = pd.concat([df_novalidos_total,df_pre_novalidos_2])
df_novalidos_total_total.shape

(5318, 12)

In [74]:
engine_2_12.dispose()

engine_2_12 = create_engine(
    url_2_12,
    fast_executemany=True,
    pool_pre_ping=True
)

# SE CARGA LAS VENTAS VALIDAS PARA COMISIONAR 

In [75]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'VENTAS_NOVEDADES_TELECAMPUS'
schema_name_2_12 = 'AGuevara'
# Send the data to sql server
df_validos.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

# SE CARGAN LAS VENTAS NO VALIDAS PARA COMISIÓN

In [76]:
table_name_2_12 = 'VENTAS_NOVEDADES_NOVALIDAS_TELECAMPUS'
schema_name_2_12 = 'AGuevara'
# Send the data to sql server
df_novalidos_total_total.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()